# Micro-assignment 4.3: validated extraction and a tool call (movies)

Same pattern as class (Pydantic contract, validate-and-retry, a tool round trip), on a NEW theme: movies. Pydantic and the tool are plain Python, but the extraction and tool-call steps make REAL calls via `llm.chat`. Set up a provider first.

## Setup

```
pip install pydantic python-dotenv
```
Plus a provider (`pip install ollama`, or a Groq/Gemini key in `.env`).

In [ ]:
from pydantic import BaseModel, ValidationError
from typing import Literal
import json, re
import llm   # real provider wrapper (gemini/groq/ollama via .env; shipped as llm.py)

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0) if m else text

# Given for problem 5: a real tool the model can ask us to run.
RATINGS = {"Inception": "8.8", "The Matrix": "8.7", "Coco": "8.4"}
def lookup_movie_rating(title: str) -> str:
    return RATINGS.get(title, "unknown")

### 1. Schema

In [ ]:
class Movie(BaseModel):
    title: str
    year: int
    genre: Literal["action", "animation", "crime", "drama", "romance", "science fiction"]

print(Movie.model_json_schema()["required"])   # ['title', 'year', 'genre']

### 2. Valid case

In [ ]:
m = Movie.model_validate_json('{"title":"Inception","year":2010,"genre":"science fiction"}')
print(m.title, m.year, isinstance(m.year, int))   # Inception 2010 True

### 3. Malformed case

In [ ]:
try:
    Movie.model_validate_json('{"title":"Inception","year":2010,"genre":"thriller"}')
except ValidationError as e:
    print(e.errors()[0]["loc"], "->", e.errors()[0]["type"])   # ('genre',) -> literal_error

### 4. Validate and retry (real model)

In [ ]:
def extract_movie(text, max_tries=3):
    messages = [{"role": "user", "content":
        "Extract the movie as JSON with keys title (string), year (integer), and "
        "genre (one of action, animation, crime, drama, romance, science fiction). "
        "Return ONLY JSON.\n"
        f"Text: {text}"}]
    for attempt in range(1, max_tries + 1):
        raw = llm.chat(messages)                                   # REAL model call
        try:
            return Movie.model_validate_json(extract_json(raw)), attempt
        except ValidationError as e:
            messages.append({"role": "user",
                             "content": f"Invalid: {e.errors()[0]['msg']}. Use an allowed genre and return ONLY JSON."})
    raise RuntimeError("no valid movie")

movie, tries = extract_movie("Inception is a 2010 science fiction film by Christopher Nolan.")
print(movie.title, movie.year, "in", tries, "tries")

### 5. Tool call (reasoning)

In [ ]:
TOOLS = {"lookup_movie_rating": {"fn": lookup_movie_rating, "schema": {
    "name": "lookup_movie_rating",
    "description": "Look up a movie's rating by its title.",
    "parameters": {"type": "object",
                   "properties": {"title": {"type": "string"}},
                   "required": ["title"]}}}}

SYSTEM = ("You can use one tool, lookup_movie_rating(title). Reply with ONE JSON object only.\n"
          'Call:   {"tool": "lookup_movie_rating", "arguments": {"title": "Inception"}}\n'
          'Answer: {"final": "Inception is rated 8.8."}')

def run(question, cap=5):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(cap):
        raw = llm.chat(messages)
        try:
            step = json.loads(extract_json(raw))
        except json.JSONDecodeError:
            messages.append({"role": "user", "content": "Reply with ONLY one JSON object."}); continue
        if "final" in step:
            return step["final"]
        result = TOOLS[step["tool"]]["fn"](**step["arguments"])
        messages.append({"role": "assistant", "content": json.dumps(step)})
        messages.append({"role": "user", "content": f"Tool result: {result}"})
    return "stopped: cap reached"

print(run("What is the rating of Inception?"))
# The cap matters because a model can loop forever, requesting tools without
# finishing. A hard step limit makes an unbounded, costly run safe and bounded,
# the first agent reliability rail in Module 5.